# 📱 Google Play Store — Business Analytics
### Análisis completo con visualizaciones independientes
---

##  CELDA 1 — Instalación de librerías y carga de datos
> **Ejecuta esta celda primero siempre. Carga el dataset y prepara el entorno.**

In [ ]:
# ── Instalación (solo necesario en Colab) ──────────────────────────────────
!pip install openpyxl -q

# ── Librerías ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Estilo global ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family'       : 'DejaVu Sans',
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.grid'         : True,
    'grid.alpha'        : 0.3,
    'figure.dpi'        : 120,
})
PALETTE = ['#4285F4','#EA4335','#FBBC05','#34A853','#FF6D00',
           '#9C27B0','#00BCD4','#795548','#607D8B','#E91E63']

# ── Subir archivo en Colab ─────────────────────────────────────────────────
from google.colab import files
print('📂 Sube tu archivo googleplaystore_limpieza.xlsx')
uploaded = files.upload()

# ── Carga ──────────────────────────────────────────────────────────────────
df = pd.read_excel('googleplaystore_limpieza.xlsx')
df['IsFree']      = ~df['Type']
df['Last Updated'] = pd.to_datetime(df['Last Updated'])
df['Year']         = df['Last Updated'].dt.year

# ── Tabla auxiliar por categoría (usada en varias gráficas) ───────────────
cat_stats = df.groupby('Category').agg(
    num_apps   = ('App',      'count'),
    avg_rating = ('Rating',   'mean'),
    total_inst = ('Installs', 'sum'),
    avg_inst   = ('Installs', 'mean'),
    pct_free   = ('IsFree',   'mean'),
).round(2)

print(f'\n✅ Dataset cargado: {df.shape[0]:,} apps | {df.shape[1]} columnas')
print(f'   Columnas: {df.columns.tolist()}')
df.head(3)

---
##  CELDA 2 — KPIs Principales

In [ ]:
total_apps      = len(df)
free_pct        = df['IsFree'].mean() * 100
avg_rating      = df['Rating'].mean()
median_installs = df['Installs'].median()
total_installs  = df['Installs'].sum()
top_category    = df['Category'].value_counts().idxmax()
avg_price_paid  = df[~df['IsFree']]['Price'].mean()

kpis = {
    '📦 Total Apps'             : f'{total_apps:,}',
    '🆓 Apps Gratuitas'         : f'{free_pct:.1f}%',
    '⭐ Rating Promedio'         : f'{avg_rating:.2f} / 5.0',
    '📥 Instalaciones Medianas' : f'{median_installs:,.0f}',
    '🌍 Instalaciones Totales'  : f'{total_installs/1e9:.2f}B',
    '🏆 Categoría Dominante'    : top_category,
    '💰 Precio Promedio (Pago)' : f'${avg_price_paid:.2f}',
}

print('='*50)
print('       KPIs PRINCIPALES — GOOGLE PLAY STORE')
print('='*50)
for k, v in kpis.items():
    print(f'  {k:<30} {v}')
print('='*50)

---
## CELDA 3 — Gráfica 1: Top 15 Categorías por Número de Apps

In [ ]:
top15 = df['Category'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 7))
colors = [PALETTE[0] if i < 3 else '#B0BEC5' for i in range(15)]
bars = ax.barh(top15.index[::-1], top15.values[::-1],
               color=colors[::-1], edgecolor='white', height=0.7)

for bar, val in zip(bars, top15.values[::-1]):
    ax.text(bar.get_width() + 8, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9, color='#333')

ax.set_title('Top 15 Categorías por Número de Apps',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Cantidad de Apps', fontsize=11)
ax.set_xlim(0, top15.max() * 1.15)
ax.spines['left'].set_visible(False)
ax.tick_params(left=False)
plt.tight_layout()
plt.show()

print(f'\n🔍 Insight: {top15.index[0]} domina con {top15.iloc[0]:,} apps '
      f'({top15.iloc[0]/len(df)*100:.1f}% del total)')

---
##  CELDA 4 — Gráfica 2: Distribución de Ratings

In [ ]:
avg_r = df['Rating'].mean()
median_r = df['Rating'].median()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['Rating'].dropna(), bins=40,
        color=PALETTE[3], edgecolor='white', alpha=0.85)
ax.axvline(avg_r,    color='red',    linestyle='--', lw=2, label=f'Promedio: {avg_r:.2f}')
ax.axvline(median_r, color='orange', linestyle=':',  lw=2, label=f'Mediana:  {median_r:.2f}')

ax.set_title('Distribución de Ratings de Apps',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Rating', fontsize=11)
ax.set_ylabel('Número de Apps', fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

bien_valoradas = (df['Rating'] >= 4.0).sum()
print(f'\n🔍 Insight: {bien_valoradas:,} apps ({bien_valoradas/len(df)*100:.1f}%) '
      f'tienen rating ≥ 4.0 — la distribución está sesgada a la derecha (positivo).')

---
## CELDA 5 — Gráfica 3: Instalaciones Totales por Categoría (Top 10)

In [ ]:
top10_inst = cat_stats['total_inst'].sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(range(10), top10_inst.values / 1e9,
              color=PALETTE[:10], edgecolor='white', width=0.65)

ax.set_xticks(range(10))
ax.set_xticklabels([c.replace('_', ' ') for c in top10_inst.index],
                   rotation=35, ha='right', fontsize=9)
ax.set_title('Instalaciones Totales por Categoría — Top 10',
             fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Instalaciones (Billones)', fontsize=11)

for bar, val in zip(bars, top10_inst.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val/1e9:.1f}B', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

lider = top10_inst.index[0]
print(f'\n🔍 Insight: {lider} lidera con {top10_inst.iloc[0]/1e9:.1f}B instalaciones totales.')

---
## CELDA 6 — Gráfica 4: Apps Gratuitas vs de Pago

In [ ]:
type_counts = df['IsFree'].value_counts()
labels = ['Gratis', 'Pago']
vals   = [type_counts[True], type_counts[False]]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Donut
wedges, texts, autotexts = axes[0].pie(
    vals, labels=labels,
    colors=[PALETTE[0], PALETTE[1]],
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 3},
    pctdistance=0.75)
for at in autotexts:
    at.set_fontsize(13); at.set_fontweight('bold')
centre = plt.Circle((0,0), 0.5, fc='white')
axes[0].add_artist(centre)
axes[0].set_title('Proporción Gratis vs Pago', fontsize=13, fontweight='bold')

# Comparación de métricas
metrics = ['avg_rating', 'pct_free']
free_stats = df[df['IsFree']].agg({'Rating':'mean','Installs':'mean'}).rename({'Rating':'Rating','Installs':'Installs avg'})
paid_stats = df[~df['IsFree']].agg({'Rating':'mean','Installs':'mean'})

compare = pd.DataFrame({
    'Gratis': [df[df['IsFree']]['Rating'].mean(), df[df['IsFree']]['Installs'].mean()/1e6],
    'Pago'  : [df[~df['IsFree']]['Rating'].mean(), df[~df['IsFree']]['Installs'].mean()/1e6],
}, index=['Rating Promedio', 'Inst. Promedio (M)'])

x = np.arange(len(compare))
width = 0.3
axes[1].bar(x - width/2, compare['Gratis'], width, label='Gratis', color=PALETTE[0], edgecolor='white')
axes[1].bar(x + width/2, compare['Pago'],   width, label='Pago',   color=PALETTE[1], edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(compare.index, fontsize=10)
axes[1].set_title('Comparación de Métricas', fontsize=13, fontweight='bold')
axes[1].legend()

plt.suptitle('Apps Gratuitas vs de Pago', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'\n🔍 Insight: Las apps de pago tienen mayor rating promedio '
      f'({paid_stats["Rating"]:.2f} vs {free_stats["Rating"]:.2f}), '
      f'pero 100x menos instalaciones.')

---
## CELDA 7 — Gráfica 5: Rating Promedio por Categoría (Top 15)

In [ ]:
avg_r = df['Rating'].mean()
top_rated = (cat_stats[cat_stats['num_apps'] >= 50]['avg_rating']
             .sort_values(ascending=False).head(15))

fig, ax = plt.subplots(figsize=(10, 7))
colors = [PALETTE[3] if r >= avg_r else PALETTE[0] for r in top_rated.values]
bars = ax.barh(top_rated.index[::-1], top_rated.values[::-1],
               color=colors[::-1], edgecolor='white', height=0.7)

ax.axvline(avg_r, color='red', linestyle='--', lw=1.5, alpha=0.7,
           label=f'Promedio global: {avg_r:.2f}')
ax.set_xlim(3.8, 4.8)
ax.set_title('Rating Promedio por Categoría\n(categorías con ≥50 apps)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Rating Promedio', fontsize=11)
ax.legend(fontsize=9)

for bar, val in zip(bars, top_rated.values[::-1]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

best = top_rated.index[0]
print(f'\n🔍 Insight: {best} tiene el mejor rating promedio: {top_rated.iloc[0]:.2f}')

---
## CELDA 8 — Gráfica 6: Clasificación de Contenido

In [ ]:
cr = df['Content Rating'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(cr.index, cr.values,
              color=PALETTE[:len(cr)], edgecolor='white', width=0.6)

ax.set_title('Distribución por Clasificación de Contenido',
             fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Número de Apps', fontsize=11)
ax.tick_params(axis='x', rotation=20)

for bar, val in zip(bars, cr.values):
    pct = val / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{val:,}\n({pct:.1f}%)', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

top_cr = cr.index[0]
print(f'\n🔍 Insight: "{top_cr}" domina con {cr.iloc[0]:,} apps '
      f'({cr.iloc[0]/len(df)*100:.1f}%) — el mercado apunta a audiencias amplias.')

---
## CELDA 9 — Gráfica 7: Rating vs Instalaciones (Scatter)

In [ ]:
sample = (df.dropna(subset=['Rating','Installs'])
          .sample(min(3000, len(df)), random_state=42))

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(
    sample['Rating'],
    np.log10(sample['Installs'] + 1),
    alpha=0.3, c=PALETTE[0], s=20, linewidths=0
)

# Línea de tendencia
z = np.polyfit(sample['Rating'], np.log10(sample['Installs'] + 1), 1)
p = np.poly1d(z)
x_line = np.linspace(1, 5, 100)
ax.plot(x_line, p(x_line), color='red', lw=2,
        linestyle='--', label='Tendencia')

ax.set_title('Relación entre Rating e Instalaciones',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Rating', fontsize=11)
ax.set_ylabel('log₁₀(Instalaciones)', fontsize=11)

# Etiquetas eje Y legibles
yticks = [0, 2, 4, 6, 8]
ax.set_yticks(yticks)
ax.set_yticklabels(['1', '100', '10K', '1M', '100M'])
ax.legend(fontsize=10)

corr = sample['Rating'].corr(np.log10(sample['Installs'] + 1))
ax.text(0.05, 0.92, f'Correlación: {corr:.3f}',
        transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

print(f'\n🔍 Insight: Correlación Rating↔log(Installs) = {corr:.3f} — '
      f'hay una relación positiva débil: un buen rating ayuda, pero no garantiza descargas.')

---
## CELDA 10 — Gráfica 8: Tendencia de Actualizaciones por Año

In [ ]:
year_counts = (df['Year'].value_counts()
               .sort_index()
               .loc[lambda s: s.index >= 2013])

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(year_counts.index, year_counts.values,
                alpha=0.15, color=PALETTE[4])
ax.plot(year_counts.index, year_counts.values, 'o-',
        color=PALETTE[4], lw=2.5, markersize=8,
        markerfacecolor='white', markeredgewidth=2)

for x, y in zip(year_counts.index, year_counts.values):
    ax.text(x, y + 40, f'{y:,}', ha='center', fontsize=10, fontweight='bold')

ax.set_title('Apps Actualizadas por Año',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Año', fontsize=11)
ax.set_ylabel('Número de Apps', fontsize=11)
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
ax.set_ylim(0, year_counts.max() * 1.2)

plt.tight_layout()
plt.show()

peak = year_counts.idxmax()
print(f'\n🔍 Insight: {peak} fue el año pico con {year_counts[peak]:,} actualizaciones — '
      f'refleja el boom de mantenimiento activo del ecosistema Android.')

---
## CELDA 11 — Gráfica 9: Mapa de Calor de Correlaciones

In [ ]:
corr_cols = ['Rating', 'Reviews', 'Installs', 'Price', 'Size_MB']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

sns.heatmap(
    corr_matrix,
    annot=True, fmt='.3f',
    cmap='RdYlGn',
    center=0, vmin=-1, vmax=1,
    square=True, linewidths=2,
    ax=ax, cbar_kws={'shrink': 0.8}
)
ax.set_title('Mapa de Correlaciones entre Variables Numéricas',
             fontsize=14, fontweight='bold', pad=15)

plt.tight_layout()
plt.show()

max_corr_pair = ('Reviews', 'Installs')
val = corr_matrix.loc[max_corr_pair[0], max_corr_pair[1]]
print(f'\n🔍 Insight: La correlación más fuerte es {max_corr_pair[0]}↔{max_corr_pair[1]} '
      f'({val:.3f}) — más reseñas = más descargas (o viceversa).')
print(f'   El Precio tiene correlación casi nula con todo: no afecta la popularidad.')

---
## CELDA 12 — Gráfica 10: Instalaciones Promedio por Categoría (Top 10)

In [ ]:
top10_avg = (cat_stats[cat_stats['num_apps'] >= 20]['avg_inst']
             .sort_values(ascending=False).head(10))

fig, ax = plt.subplots(figsize=(11, 6))
colors = [PALETTE[i % len(PALETTE)] for i in range(10)]
bars = ax.barh(top10_avg.index[::-1], top10_avg.values[::-1] / 1e6,
               color=colors[::-1], edgecolor='white', height=0.65)

for bar, val in zip(bars, top10_avg.values[::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val/1e6:.1f}M', va='center', fontsize=9, fontweight='bold')

ax.set_title('Instalaciones Promedio por App — Top 10 Categorías\n(categorías con ≥20 apps)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Instalaciones Promedio (Millones)', fontsize=11)
ax.spines['left'].set_visible(False)
ax.tick_params(left=False)

plt.tight_layout()
plt.show()

top = top10_avg.index[0]
print(f'\n🔍 Insight: Una app en {top} puede esperar ~{top10_avg.iloc[0]/1e6:.0f}M '
      f'instalaciones en promedio — muy por encima del resto.')

---
## CELDA 13 — Tabla Resumen Final (Para copiar a Power BI)

In [ ]:
print('═'*60)
print('   RESUMEN EJECUTIVO — GOOGLE PLAY STORE')
print('═'*60)

print('\n📌 Top 5 por nº de apps:')
print(cat_stats['num_apps'].sort_values(ascending=False).head(5).to_string())

print('\n📌 Top 5 por instalaciones totales:')
t = cat_stats['total_inst'].sort_values(ascending=False).head(5)
print((t/1e9).round(2).astype(str).add(' B').to_string())

print('\n📌 Top 5 por rating promedio (≥50 apps):')
print(cat_stats[cat_stats['num_apps']>=50]['avg_rating']
      .sort_values(ascending=False).head(5).to_string())

print('\n📌 Gratis vs Pago:')
gp = df.groupby('IsFree').agg(
    apps=('App','count'),
    rating=('Rating','mean'),
    installs_prom=('Installs','mean')
).rename(index={True:'Gratis', False:'Pago'}).round(2)
print(gp.to_string())

print('\n✅ Todos los análisis completados.')